# 주제 ① 사출성형 — 추가 진단 (02_diagnosis_deep)

**목적**: `01_data_quality.ipynb`에서 "추정"으로 남긴 주장을 검증하고, 다루지 않은 축(타깃 정의, 비가동 블록 정밀 정의,
시간 누수 크기, labeled↔unlabeled 관계, 자기상관, cn7·rg3 결합 가능성, 이상치 유형 규칙)을 추가로 진단한다.
심사기준 1번(데이터 이해 및 진단)에 대응하며, **모델링 경쟁이 아니라 데이터 진단**이다. 타깃 정의 비교에 쓰는 로지스틱
회귀는 성능 경쟁용이 아니라 "어떤 타깃 정의가 안정적인 학습을 가능하게 하는가"를 보기 위한 소형 실험이다.

**구성**: 계획서(`.claude/plans/2026-09-26_01_03_followup_plan.md` A-2) 번호 순.
① 쌍 구조 규명 · ② 비가동 블록 정밀 정의 · ③ 드리프트·시간 누수 · ④ labeled↔unlabeled 매칭 ·
⑤ 변수 성격 전수 점검 · ⑥ 샷 간 자기상관 · ⑦ cn7·rg3 결합 가능성 · ⑧ 이상치 유형 규칙

각 섹션은 **가설 → 실험 → 결과 → 모델 단계 반영** 순으로 서술한다.


In [1]:
import sys, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
import data_quality as dq
import pairs, drift, outliers
FIG = ROOT / "figures"; FIG.mkdir(exist_ok=True)
plt.rcParams["font.family"] = "Malgun Gothic"; plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 80)
def save(name): plt.tight_layout(); plt.savefig(FIG / name, dpi=110); plt.close(); print("saved", name)


In [2]:
dfs = dq.load_all()
for k, d in dfs.items():
    print(f"{k:14s} shape={d.shape}")


labeled_cn7    shape=(1211, 25)
labeled_rg3    shape=(1182, 25)
unlabeled_cn7  shape=(35239, 24)
unlabeled_rg3  shape=(35941, 24)


## ① 쌍 구조 규명

**가설**: labeled의 모든 X가 정확히 2회 등장한다는 것은 01에서 확인했다. 그런데 이것이 (a) 24변수가 **완전히 동일**해서인지
아니면 (b) `row_key`가 8자리로 반올림해서 우연히 같아 보이는 것인지는 미확인이다. 또한 라벨 충돌 쌍이 특정 변수 조건에
몰려 있는지, 충돌 쌍 안에서 불량이 항상 같은 순서(첫 행 또는 둘째 행)에 오는지도 미확인이다.

**실험**: `pairs.pair_table`로 키별 그룹(쌍/예외)을 만들고 24변수 **반올림 없는 완전 일치** 여부를 직접 비교한다.
인접하지 않은 쌍의 위치를 확인한다. `pairs.mannwhitney_conflict_vs_agree`로 충돌 쌍과 일치 쌍의 변수별 분포 차이를
Mann-Whitney U로 검정한다. `pairs.order_bias_test`로 충돌 쌍 내 불량 행의 순서 편향을 이항검정한다. 마지막으로
`pairs.target_variants` + `pairs.cv_f1_by_target`으로 타깃 정의 3안(max/mean≥0.5/충돌 제외)의 양성 수와
GroupKFold(X-키) 5-fold 로지스틱(class_weight='balanced') F1 분포를 비교한다.


In [3]:
pt_cn7 = pairs.pair_table(dfs["labeled_cn7"])
pt_rg3 = pairs.pair_table(dfs["labeled_rg3"])
for name, pt in [("cn7", pt_cn7), ("rg3", pt_rg3)]:
    print(f"===== {name}")
    print("group_size:", pt["group_size"].value_counts().to_dict())
    print("pattern:", pt["pattern"].value_counts().to_dict())
    print("exact_equal(그룹크기2):", pt.loc[pt.group_size == 2, "exact_equal"].value_counts().to_dict())


===== cn7
group_size: {2: 605, 1: 1}
pattern: {'00': 591, '01': 11, '11': 3, 'single': 1}
exact_equal(그룹크기2): {True: 605}
===== rg3
group_size: {2: 591}
pattern: {'00': 566, '01': 25}
exact_equal(그룹크기2): {True: 591}


In [4]:
# 인접 여부(24변수 완전일치 쌍 기준)
for name, pt in [("cn7", pt_cn7), ("rg3", pt_rg3)]:
    p2 = pt[pt.group_size == 2]
    n_non_adj = int((p2["adjacent"] == False).sum())
    print(f"{name}: 비인접 쌍 {n_non_adj}/{len(p2)} ({n_non_adj/len(p2):.1%}), gap 분포:")
    print(p2["gap"].value_counts().sort_index().to_dict())


cn7: 비인접 쌍 35/605 (5.8%), gap 분포:
{1.0: 570, 2.0: 20, 3.0: 12, 4.0: 2, 5.0: 1}
rg3: 비인접 쌍 17/591 (2.9%), gap 분포:
{1: 574, 2: 8, 3: 9}


In [5]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (name, pt) in zip(axes, [("cn7", pt_cn7), ("rg3", pt_rg3)]):
    p2 = pt[pt.group_size == 2]
    ax.hist(p2["gap"], bins=np.arange(0.5, p2["gap"].max() + 1.5, 1), color="steelblue", edgecolor="white")
    ax.set_title(f"{name}: 쌍 인덱스 간격(gap) 분포"); ax.set_xlabel("gap (idx2-idx1)")
save("02_pair_gap.png")


saved 02_pair_gap.png


In [6]:
# 비인접 쌍의 위치(파일 내 어디에 몰려 있는가)
for name, pt in [("cn7", pt_cn7), ("rg3", pt_rg3)]:
    p2 = pt[pt.group_size == 2]
    non_adj = p2[p2["adjacent"] == False].sort_values("idx1")
    if len(non_adj):
        print(f"{name}: 비인접 쌍 idx1 범위 [{non_adj['idx1'].min()}, {non_adj['idx1'].max()}] "
              f"(파일 전체 {pt['idx1'].min()}~{int(pt['idx2'].max())})")
    else:
        print(f"{name}: 비인접 쌍 없음")


cn7: 비인접 쌍 idx1 범위 [295, 1190] (파일 전체 0~1210)
rg3: 비인접 쌍 idx1 범위 [1351, 2268] (파일 전체 1211~2392)


In [7]:
print("=== 충돌 쌍 vs 일치 쌍 변수별 Mann-Whitney (cn7, p오름차순 상위 8) ===")
mw_cn7 = pairs.mannwhitney_conflict_vs_agree(dfs["labeled_cn7"], pt_cn7)
mw_cn7.head(8)


=== 충돌 쌍 vs 일치 쌍 변수별 Mann-Whitney (cn7, p오름차순 상위 8) ===


,n_conflict,n_agree,U,p,effect_r,p_bonf
var,,,,,,
Mold_Temperature_3,11,594,5851.0,0.000007,-0.790940,0.000157
Mold_Temperature_4,11,594,5822.5,0.000008,-0.782216,0.000199
Filling_Time,11,594,5668.5,0.000024,-0.735078,0.000580
Plasticizing_Position,11,594,5633.5,0.000025,-0.724365,0.000610
Injection_Time,11,594,5610.0,0.000038,-0.717172,0.000904
Max_Injection_Speed,11,594,930.0,0.000043,0.715335,0.001022
Clamp_Close_Time,11,594,5279.5,0.000056,-0.616009,0.001349
Plasticizing_Time,11,594,1141.5,0.000213,0.650597,0.005107


In [8]:
print("=== 충돌 쌍 vs 일치 쌍 변수별 Mann-Whitney (rg3, p오름차순 상위 8) ===")
mw_rg3 = pairs.mannwhitney_conflict_vs_agree(dfs["labeled_rg3"], pt_rg3)
mw_rg3.head(8)


=== 충돌 쌍 vs 일치 쌍 변수별 Mann-Whitney (rg3, p오름차순 상위 8) ===


,n_conflict,n_agree,U,p,effect_r,p_bonf
var,,,,,,
Barrel_Temperature_5,25,566,9034.0,0.018246,-0.276890,0.437907
Plasticizing_Time,25,566,7956.0,0.290612,-0.124523,1.000000
Cycle_Time,25,566,7747.0,0.351558,-0.094982,1.000000
Clamp_Close_Time,25,566,7738.5,0.387304,-0.093781,1.000000
Barrel_Temperature_3,25,566,6375.0,0.397362,0.098940,1.000000
Hopper_Temperature,25,566,6429.0,0.439325,0.091307,1.000000
Max_Screw_RPM,25,566,7626.0,0.473980,-0.077880,1.000000
Average_Screw_RPM,25,566,7513.0,0.551249,-0.061908,1.000000


In [9]:
print("cn7 Bonferroni 유의(p_bonf<0.05) 변수 수:", int((mw_cn7["p_bonf"] < 0.05).sum()), "/", len(mw_cn7))
print("rg3 Bonferroni 유의(p_bonf<0.05) 변수 수:", int((mw_rg3["p_bonf"] < 0.05).sum()), "/", len(mw_rg3))


cn7 Bonferroni 유의(p_bonf<0.05) 변수 수: 10 / 24
rg3 Bonferroni 유의(p_bonf<0.05) 변수 수: 0 / 24


In [10]:
print("=== 충돌 쌍 내 불량 행 순서 편향 ===")
ob_cn7 = pairs.order_bias_test(pt_cn7)
ob_rg3 = pairs.order_bias_test(pt_rg3)
print("cn7:", ob_cn7)
print("rg3:", ob_rg3)


=== 충돌 쌍 내 불량 행 순서 편향 ===
cn7: {'n_conflict': 11, 'fail_first': 10, 'fail_second': 1, 'binom_p': np.float64(0.01171875)}
rg3: {'n_conflict': 25, 'fail_first': 23, 'fail_second': 2, 'binom_p': np.float64(1.9431114196777344e-05)}


**결과**
- 24변수를 반올림 없이 비교해도 모든 쌍이 **완전 일치**한다(cn7 605쌍, rg3 591쌍 전부 `exact_equal=True`). 8자리 반올림
  키가 우연히 같아 보이는 경우는 없다 → 진짜 "같은 입력이 2행으로 기록"된 구조다.
- 비인접 쌍 비율은 cn7 5.8%(35/605, gap 최대 5), rg3는 전부 인접(0%)이다. cn7 비인접 쌍은 특정 구간(행
  약 295~1192)에 몰려 있고, 서로 다른 두 쌍의 인덱스가 교차(예: 401·403 쌍과 402·404 쌍)하는 형태 → 그 구간에서
  기록 순서가 인터리브된 것으로 추정.
- 충돌 쌍 vs 일치 쌍 Mann-Whitney: **cn7은 Bonferroni 보정 후에도 24개 중 10개 변수**(주로 `Mold_Temperature_3/4`,
  `Filling_Time`, `Plasticizing_Position`, `Injection_Time`)**가 유의(p_bonf<0.05)**하다 → cn7 충돌 쌍은 무작위로
  흩어진 게 아니라 드리프트 구간(③ 참고)에 몰려 있다는 뜻. **rg3는 24개 중 유의한 변수가 0개**(최소 p_bonf=0.44) →
  rg3 충돌은 특정 조건과 무관하게 전 구간에 고르게 나타나는 순수한 라벨 노이즈에 가깝다.
- 순서 편향: 충돌 쌍에서 불량이 **첫 번째 행에 오는 경우가 압도적으로 많다**(cn7 10/11, 이항검정 p≈0.012; rg3 23/25,
  p≈2e-05). 우연이라면 50%에 가까워야 하는데 크게 벗어난다 → "먼저 기록된 샷이 불합격, 재검사/재작업 후 두 번째
  기록이 합격"인 수집 관행일 가능성이 높다(추정). 이는 3안(충돌 제외) 외에 **"첫 행 라벨을 신뢰"하는 4번째 타깃 정의
  후보**를 시사한다.

**모델 단계 반영**: cn7은 GroupKFold와 별개로 **시간 블록 split을 반드시 병행**해야 한다(충돌이 드리프트 구간에
몰려 있어 랜덤 split은 이 구간을 학습에도 노출시켜 과대평가를 유발). rg3는 조건-무관 라벨 노이즈이므로 피처 튜닝보다
**라벨 신뢰도 자체를 모델 성능 상한으로 보고**해야 한다. 순서 편향은 참고용 가설로 리포트에만 남기고(표본 11·25건으로
추가 검증 없이 규칙화하기엔 근거가 약함), 실제 타깃 정의는 아래 F1 비교로 결정한다.


In [11]:
variants_cn7, meta_cn7 = pairs.target_variants(dfs["labeled_cn7"])
variants_rg3, meta_rg3 = pairs.target_variants(dfs["labeled_rg3"])
print("cn7 meta:", meta_cn7)
print("rg3 meta:", meta_rg3)
summary = []
for name, files in [("cn7", variants_cn7), ("rg3", variants_rg3)]:
    for tname, (X, y, g) in files.items():
        summary.append({"file": name, "target": tname, "n_rows": len(y), "n_pos": int(y.sum())})
pd.DataFrame(summary)


cn7 meta: {'group_pos': 14, 'group_soft_pos': 8.5, 'group_pure_pos': 3, 'n_groups': 606, 'n_conflict_groups': 11}
rg3 meta: {'group_pos': 25, 'group_soft_pos': 12.5, 'group_pure_pos': 0, 'n_groups': 591, 'n_conflict_groups': 25}


,file,target,n_rows,n_pos
0,cn7,max,1211,28
1,cn7,mean_ge_0.5,1211,28
2,cn7,exclude_conflict,1189,6
3,rg3,max,1182,50
4,rg3,mean_ge_0.5,1182,50
5,rg3,exclude_conflict,1132,0


In [12]:
f1_cn7 = pairs.cv_f1_by_target(dfs["labeled_cn7"])
f1_rg3 = pairs.cv_f1_by_target(dfs["labeled_rg3"])
print("=== cn7 F1 (target별 median/mean/std/count) ===")
display(f1_cn7.groupby("target")["f1"].agg(["median", "mean", "std", "count"]))
print("=== rg3 F1 ===")
display(f1_rg3.groupby("target")["f1"].agg(["median", "mean", "std", "count"]))
print(f1_rg3.loc[f1_rg3["note"] != "", ["target", "note"]].drop_duplicates())


=== cn7 F1 (target별 median/mean/std/count) ===


,median,mean,std,count
target,,,,
exclude_conflict,1.000000,0.600000,0.547723,5
max,0.153846,0.184615,0.200591,5
mean_ge_0.5,0.153846,0.184615,0.200591,5


=== rg3 F1 ===


,median,mean,std,count
target,,,,
exclude_conflict,NaN,NaN,NaN,0
max,0.078431,0.060958,0.059168,5
mean_ge_0.5,0.078431,0.060958,0.059168,5


              target                    note
10  exclude_conflict  양성 또는 음성 표본 0개 - 실행 불가


In [13]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (name, f1df) in zip(axes, [("cn7", f1_cn7), ("rg3", f1_rg3)]):
    d = f1df.dropna(subset=["f1"])
    if len(d):
        sns.stripplot(data=d, x="target", y="f1", ax=ax, size=8, color="steelblue")
        sns.boxplot(data=d, x="target", y="f1", ax=ax, showcaps=True, boxprops={"facecolor": "none"}, showfliers=False)
    ax.set_title(f"{name}: 타깃 정의별 GroupKFold F1"); ax.set_ylim(-0.05, 1.05)
    ax.tick_params(axis="x", rotation=15)
save("02_target_f1_dist.png")


saved 02_target_f1_dist.png


**결과**
- 그룹 크기가 항상 1~2이므로 `max`와 `mean≥0.5`는 **이진 결정이 수학적으로 동일**하다(둘 다 충돌 쌍을 양성으로 만든다).
  차이는 "기대 양성 수"에서만 드러난다: cn7 그룹 단위 고유 불량 14개(=max 기준) vs 8.5개(=mean 기준, 충돌을 0.5로
  카운트), rg3는 25개 vs 12.5개.
- `충돌 제외`는 rg3에서 **양성이 0개가 되어 분류 자체가 불가능**하다(rg3 불량 25개 전부가 충돌 쌍이므로). cn7은
  양성 6행(순수 불량 그룹 3개×2행)만 남아 5-fold 중 다수 fold에 양성이 1개뿐이라 F1이 fold마다 0 또는 1로 극단적으로
  흔든다(median 1.0, mean 0.60, std 0.55, n=5).
- `max`(=`mean≥0.5`) 기준 F1은 cn7 median 0.154(mean 0.185, std 0.20), rg3 median 0.078(mean 0.061, std 0.059) —
  두 정의 모두 절대 수준이 낮고 fold 간 편차가 크다.

**모델 단계 반영**: (1) rg3는 `충돌 제외`를 쓸 수 없다(양성 0) → 반드시 `max`(불량 의심 포함) 정의를 쓰거나, 충돌
쌍을 "판정 불가(모델 평가 제외, 학습에는 약한 양성으로만 사용)"로 별도 처리해야 한다. (2) cn7의 `충돌 제외` F1이
높아 보이는 것은 **표본이 너무 적어(양성 6개) 나온 착시**이므로 "충돌 제외가 더 좋은 정의"라고 해석하면 안 된다 —
표에 그대로 신뢰구간 없이 보고하지 않는다. (3) 모델 단계 기본값은 **`max` 정의 + GroupKFold(X-키) + 시간 블록 split
병행**으로 확정하고, `충돌 제외`는 "라벨이 확실한 샷만의 상한선 참고용"으로만 부기한다.


## ② 비가동 블록 정밀 정의

**가설**: 01에서는 6개 마커 변수가 "한 값에 고정"된 행을 비가동으로 추정했다. 그 고정값이 실제로 파일의 **최솟값**인지
(설비 정지 시 0으로 리셋되는 물리적 신호라는 가설), 비가동 행에서 **나머지 18변수**는 어떤 분포를 보이는지(전부
고정인지, 일부만 고정인지), 마커 4/5/6개 일치 기준에 따라 비율이 어떻게 달라지는지, labeled에도 부분 고정 행이
있는지는 미확인이었다.

**실험**: `outliers.marker_mode_is_min`, `outliers.idle_sensitivity_table`, `outliers.idle_rows_other_var_profile`을
unlabeled·labeled 양쪽에 적용한다.


In [14]:
print("=== 마커 6변수 최빈값 == 최솟값? (unlabeled) ===")
print("cn7:"); display(outliers.marker_mode_is_min(dfs["unlabeled_cn7"]))
print("rg3:"); display(outliers.marker_mode_is_min(dfs["unlabeled_rg3"]))


=== 마커 6변수 최빈값 == 최솟값? (unlabeled) ===
cn7:


,mode,min,mode_eq_min
var,,,
Max_Screw_RPM,-0.967689,-0.967689,True
Average_Screw_RPM,-0.969766,-0.969766,True
Average_Back_Pressure,-0.937754,-0.937754,True
Barrel_Temperature_1,-0.969455,-0.969455,True
Mold_Temperature_3,-0.952199,-0.952199,True
Mold_Temperature_4,-0.936266,-0.936266,True


rg3:


,mode,min,mode_eq_min
var,,,
Max_Screw_RPM,-1.179145,-1.179145,True
Average_Screw_RPM,-1.297009,-1.297009,True
Average_Back_Pressure,-1.154268,-1.154268,True
Barrel_Temperature_1,-1.310547,-1.310547,True
Mold_Temperature_3,-1.276966,-1.276966,True
Mold_Temperature_4,-1.225083,-1.225083,True


In [15]:
print("=== 마커 일치 개수(>=4/>=5/==6) 비율: unlabeled ===")
print("cn7:"); display(outliers.idle_sensitivity_table(dfs["unlabeled_cn7"]))
print("rg3:"); display(outliers.idle_sensitivity_table(dfs["unlabeled_rg3"]))
print("=== 마커 일치 개수 비율: labeled (부분 고정 확인) ===")
print("cn7:"); display(outliers.idle_sensitivity_table(dfs["labeled_cn7"]))
print("rg3:"); display(outliers.idle_sensitivity_table(dfs["labeled_rg3"]))


=== 마커 일치 개수(>=4/>=5/==6) 비율: unlabeled ===
cn7:


,min_markers,n_rows,share
0,4,18151,0.515083
1,5,18151,0.515083
2,6,18151,0.515083


rg3:


,min_markers,n_rows,share
0,4,13154,0.365989
1,5,13154,0.365989
2,6,13154,0.365989


=== 마커 일치 개수 비율: labeled (부분 고정 확인) ===
cn7:


,min_markers,n_rows,share
0,4,42,0.034682
1,5,2,0.001652
2,6,0,0.000000


rg3:


,min_markers,n_rows,share
0,4,42,0.035533
1,5,8,0.006768
2,6,2,0.001692


In [16]:
print("=== 비가동 행(6개 마커 전부 고정)에서 나머지 18변수 분포: cn7 ===")
prof_cn7 = outliers.idle_rows_other_var_profile(dfs["unlabeled_cn7"])
display(prof_cn7[["mean", "std", "mean_all", "std_all", "nunique"]])


=== 비가동 행(6개 마커 전부 고정)에서 나머지 18변수 분포: cn7 ===


,mean,std,mean_all,std_all,nunique
Injection_Time,-0.838105,0.185585,1.032373e-16,1.000014,2.0
Filling_Time,0.727351,0.686706,-8.129936e-16,1.000014,4.0
Plasticizing_Time,-0.959868,0.021143,-2.580932e-17,1.000014,19.0
Cycle_Time,-0.950626,0.092969,7.742797e-16,1.000014,49.0
Clamp_Close_Time,-0.964657,0.137356,-7.742797e-17,1.000014,15.0
Cushion_Position,-0.970275,0.001475,-1.290466e-16,1.000014,42.0
Plasticizing_Position,-0.964963,0.062394,-5.161864e-17,1.000014,9.0
Clamp_Open_Position,-0.434021,0.122185,1.496941e-15,1.000014,38.0
Max_Injection_Speed,-0.887415,0.351028,-1.290466e-16,1.000014,100.0
Max_Injection_Pressure,-0.945610,0.044910,-3.097119e-16,1.000014,71.0


In [17]:
fig, ax = plt.subplots(figsize=(11, 4.5))
prof_cn7_sorted = prof_cn7.sort_values("std")
ax.bar(range(len(prof_cn7_sorted)), prof_cn7_sorted["std"], color="steelblue")
ax.set_xticks(range(len(prof_cn7_sorted))); ax.set_xticklabels(prof_cn7_sorted.index, rotation=75, fontsize=8)
ax.axhline(0.1, ls="--", c="gray")
ax.set_title("cn7 비가동 행 내 나머지 18변수의 표준편차(작을수록 그 값도 고정에 가까움)")
save("02_idle_profile.png")


saved 02_idle_profile.png


**결과**
- 6개 마커 변수 모두 **최빈값(고정값) == 파일 최솟값**(cn7·rg3 공통). 표준화 전 원본값이 물리적 하한(0, 정지 상태)에서
  고정된다는 01의 추정을 뒷받침한다.
- 마커 일치 개수별 비율이 unlabeled에서는 **>=4, >=5, ==6이 모두 동일한 비율**(cn7 51.5%, rg3 36.6%)이다 → 4개 이상
  일치하는 순간 항상 6개 전부 일치한다(부분 고정이 사실상 없는 **all-or-nothing** 현상). "일부 마커만 고정"인 애매한
  중간 상태는 unlabeled에 존재하지 않는다.
- **labeled에는 부분 고정이 실제로 존재한다**: cn7은 마커 4개 이상 일치 42행(3.5%), 5개 이상 2행(0.17%), 6개 전부
  일치 0행. **rg3는 6개 전부 일치가 2행(0.17%) 있다** — 01의 "labeled에는 비가동 블록이 없다"는 단정은 **부분적으로
  틀렸다**: rg3에는 완전 일치 기준으로도 극소수(2건)가 존재한다.
- 비가동 행에서 나머지 18변수 중 **속도·압력·사이클·위치 계열(Cycle_Time, Cushion_Position, Max_Injection_Pressure 등,
  std 0.001~0.09)은 거의 완전히 고정**되는 반면, **온도 계열(Barrel_Temperature_2~6, Hopper_Temperature, std
  0.35~0.94)은 상당히 퍼져 있다**(nunique 24~186). 설비가 멈춰도 배럴·금형은 서서히 냉각되므로 온도만 서서히
  변한다는 물리적 설명과 일치한다.

**모델 단계 반영**: (1) 비가동 마스크는 "4개 이상 일치"로 완화해도 unlabeled에서는 결과가 동일하므로 현재 6개 기준
그대로 유지해도 무방하다. (2) **labeled에도 극소수(rg3 2건) 완전 고정 행이 있으므로, 학습에서 완전히 제외하기보다
"비가동 근접도" 연속 플래그(마커 일치 개수, 온도 std 등)를 파생 변수로 남겨 완전 제거의 부작용(라벨 있는 표본 손실)을
피한다. (3) 온도 계열은 비가동 중에도 정보가 남아 있으므로 "직전 비가동 지속시간 추정"과 같은 파생 변수 후보가 된다.


## ③ 드리프트 · 시간 누수 검증

**가설**: 01은 cn7의 `Mold_Temperature_3/4` 단일 변수 AUC 0.89를 "불량 조건이 아니라 불량이 난 시기를 가리키는
시간 누수성 상관"으로 추정했지만, 실제로 시간 블록 split에서 AUC가 얼마나 떨어지는지는 계산하지 않았다. 또한
"충전 계열 동시 극단 10행"이 불량 밀집 구간(행 99~120)과 같은 사건인지도 미확인이었다.

**실험**: `drift.block_stats`로 5개 블록의 평균·KS·PSI(블록0 대비)를 계산해 드리프트 크기를 정량화한다.
`drift.auc_by_split`로 같은 변수의 랜덤 5-fold AUC(평균)와 "블록0(행 0~241, 불량 밀집 행 99~120 포함)을 test로 쓰는
시간 분할" AUC를 비교한다. `drift.simultaneous_extreme_rows`로 사출·충전 계열 4변수 동시 극단 행을 다시 찾아 위치를
확인한다.


In [18]:
X_cn7, y_cn7 = dq.split_xy(dfs["labeled_cn7"])
X_rg3, y_rg3 = dq.split_xy(dfs["labeled_rg3"])
bs_cn7 = drift.block_stats(X_cn7, n_blocks=5)
bs_rg3 = drift.block_stats(X_rg3, n_blocks=5)
print("=== cn7 블록 드리프트 상위 8 (max_psi 기준) ===")
display(bs_cn7[["max_psi", "min_ks_p"]].head(8))
print("=== rg3 블록 드리프트 상위 8 ===")
display(bs_rg3[["max_psi", "min_ks_p"]].head(8))


=== cn7 블록 드리프트 상위 8 (max_psi 기준) ===


,max_psi,min_ks_p
var,,
Injection_Time,10.012531,9.878640e-125
Filling_Time,9.419629,2.300669e-113
Plasticizing_Position,8.296683,5.534566e-145
Plasticizing_Time,8.245065,5.534566e-145
Hopper_Temperature,8.165552,6.540706e-129
Mold_Temperature_4,7.960629,5.534566e-145
Mold_Temperature_3,7.700707,5.534566e-145
Max_Back_Pressure,6.754633,1.291164e-139


=== rg3 블록 드리프트 상위 8 ===


,max_psi,min_ks_p
var,,
Mold_Temperature_3,8.457720,2.238825e-141
Mold_Temperature_4,8.352955,2.238825e-141
Max_Injection_Speed,8.064610,5.392304e-81
Average_Back_Pressure,5.981290,4.806828e-67
Hopper_Temperature,5.243083,3.322231e-71
Plasticizing_Time,4.779699,1.899894e-62
Cushion_Position,4.545154,5.352497e-48
Max_Injection_Pressure,3.364344,7.646788e-34


In [19]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (name, bs) in zip(axes, [("cn7", bs_cn7), ("rg3", bs_rg3)]):
    top = bs.head(8)
    ax.barh(top.index[::-1], top["max_psi"][::-1], color="indianred")
    ax.set_title(f"{name}: 블록 간 최대 PSI(블록0 대비) 상위 8"); ax.set_xlabel("max PSI")
save("02_drift_blocks.png")


saved 02_drift_blocks.png


In [20]:
test_idx_cn7 = X_cn7.index[0:len(X_cn7) // 5]
test_idx_rg3 = X_rg3.index[0:len(X_rg3) // 5]
print("cn7 시간분할 test 구간:", test_idx_cn7.min(), "~", test_idx_cn7.max(), f"(n={len(test_idx_cn7)})")
res = []
for var in ["Mold_Temperature_3", "Mold_Temperature_4"]:
    res.append(drift.auc_by_split(X_cn7, y_cn7, var, test_idx_cn7))
res.append(drift.auc_by_split(X_rg3, y_rg3, "Barrel_Temperature_5", test_idx_rg3))
pd.DataFrame(res)


cn7 시간분할 test 구간: 0 ~ 241 (n=242)


,var,auc_random_mean,auc_random_std,auc_time_block,n_time_test,n_time_test_pos
0,Mold_Temperature_3,0.898769,0.031774,0.605752,242,17
1,Mold_Temperature_4,0.894012,0.027384,0.657386,242,17
2,Barrel_Temperature_5,0.632146,0.106562,0.510776,236,4


In [21]:
fig, ax = plt.subplots(figsize=(7, 4.5))
d = pd.DataFrame(res)
x = np.arange(len(d))
ax.bar(x - 0.18, d["auc_random_mean"], width=0.36, yerr=d["auc_random_std"], label="랜덤 5-fold AUC(평균)", color="steelblue")
ax.bar(x + 0.18, d["auc_time_block"], width=0.36, label="시간블록 AUC(블록0=test)", color="indianred")
ax.set_xticks(x); ax.set_xticklabels(d["var"], rotation=10); ax.axhline(0.5, ls="--", c="gray")
ax.set_ylim(0, 1); ax.legend(); ax.set_title("단일 변수 AUC: 랜덤 split vs 시간 블록 split")
save("02_auc_random_vs_time.png")


saved

 02_auc_random_vs_time.png


In [22]:
cols = ["Injection_Time", "Filling_Time", "Max_Injection_Pressure", "Max_Injection_Speed"]
ext_cn7 = drift.simultaneous_extreme_rows(X_cn7, cols, z=3.0, k=2)
print("cn7 4변수 동시 극단 행:", ext_cn7["idx"].tolist())
print("불량 여부:", y_cn7.loc[ext_cn7["idx"]].tolist())
in_block = ext_cn7["idx"].between(99, 120)
print(f"99~120 구간 안: {int(in_block.sum())}행, 구간 밖: {int((~in_block).sum())}행")


cn7 4변수 동시 극단 행: [115, 116, 117, 118, 119, 120, 1207, 1208, 1209, 1210]
불량 여부: [1, 1, 1, 1, 1, 1, 0, 0, 0, 0]
99~120 구간 안: 6행, 구간 밖: 4행


**결과**
- 드리프트 크기: cn7·rg3 **둘 다** 여러 변수에서 블록 간 PSI가 5~10(통상 PSI>0.25면 "큰 변화"로 간주하는 기준을
  훌쩍 넘는다)에 달하고 KS p-value가 사실상 0에 가깝다 → 01에서 "cn7만 드리프트가 크다"고 서술한 것과 달리
  **rg3도 여러 변수(`Mold_Temperature_3/4`, `Max_Injection_Speed`, `Average_Back_Pressure` 등)에서 강한 드리프트가
  있다**. 다만 rg3는 불량이 전 구간에 분산돼 있어 드리프트가 불량 예측에 직접 누수로 이어지지 않을 뿐이다.
- cn7 `Mold_Temperature_3`: 랜덤 5-fold AUC 평균 **0.899**(std 0.032, 전체 AUC 0.894와 거의 동일) vs 시간블록(블록0을
  test로) AUC **0.606** — **누수 크기 약 0.29~0.30**. `Mold_Temperature_4`도 0.894→0.657(누수 약 0.24). 시간 정보를
  차단하면 판별력이 절반 가까이 사라진다 → 01의 "시간 누수성 상관" 추정이 **정량적으로 확인**됐다.
- rg3 `Barrel_Temperature_5`(01에서 확인한 rg3 최고 단일 AUC 변수)는 랜덤 0.632(std 0.107) vs 시간블록 0.511 —
  절대 수준 자체가 낮아 누수 여부와 무관하게 예측력이 거의 없다(0.5에 근접).
- "충전 계열 동시 극단 10행"은 **6행(115~120)만 불량 밀집 구간(99~120)과 같은 사건**이고, **나머지 4행(1207~1210,
  파일 맨 끝)은 전부 양품인 별개의 극단 이벤트**다. 즉 01의 "동시 극단=공정 이상 신호"라는 해석은 **절반만 맞다**:
  같은 사건(115~120) 안에서는 극단값과 불량이 같이 나타나지만, 파일 끝(1207~1210)에서는 같은 극단값 패턴이
  불량과 무관하게 나타난다 → 이 4변수 동시극단은 "불량"이 아니라 "설비 정지/재가동 직전 마지막 샷들"과 같은
  **비정상 종료 패턴**을 가리킬 가능성이 있다(추정).

**모델 단계 반영**: (1) cn7·rg3 모두 랜덤 split만으로 평가하면 시간 누수로 성능이 과대평가되므로 **시간 블록
split(예: 블록0을 test)을 GroupKFold와 나란히 반드시 보고**한다. (2) `Mold_Temperature_3/4`처럼 랜덤-시간 AUC
격차가 큰 변수는 "판별력이 크다"가 아니라 "시간과 얽혀 있다"로 해석하고, 모델에 넣을 때 **행 순서 기반 파생변수
(예: 이동평균 대비 편차)로 변환**해 절대 수준이 아닌 상대 변화를 신호로 쓰는 것을 검토한다. (3) "4변수 동시 극단"
규칙은 파일 끝 근처 행에도 반응하므로, 규칙 기반 예측(⑧)에서 오탐 원인으로 남긴다.


## ④ labeled <-> unlabeled 근사 매칭

**가설**: labeled와 unlabeled는 스케일이 달라(파일별 z-score) 값으로 직접 비교할 수 없다. 원본 인덱스 범위도 겹치지
않는다(01에서 확인). 그러나 **같은 샷이 두 파일에 모두 기록**됐다면(예: labeled가 unlabeled의 부분집합을 다시
표준화한 것이라면) unlabeled를 사전학습·이상탐지에 쓸 때 test 행이 섞이는 누수가 생긴다.

**실험**: 파일 내부 순위(percentile rank)로 정규화한 24차원 벡터로 labeled 각 행의 unlabeled 최근접 이웃 거리를
구하고, 완전 무작위 벡터의 최근접 거리(기준선)와 비교한다(`pairs.rank_match_to_reference`).


In [23]:
match_cn7, rand_cn7 = pairs.rank_match_to_reference(dfs["labeled_cn7"], dfs["unlabeled_cn7"])
match_rg3, rand_rg3 = pairs.rank_match_to_reference(dfs["labeled_rg3"], dfs["unlabeled_rg3"])
from scipy.stats import ks_2samp
for name, m, r in [("cn7", match_cn7, rand_cn7), ("rg3", match_rg3, rand_rg3)]:
    stat, p = ks_2samp(m["dist"], r)
    print(f"===== {name}")
    print("labeled->unlabeled 최근접거리:", m["dist"].describe().round(3).to_dict())
    print("무작위 벡터 최근접거리(기준선):", pd.Series(r).describe().round(3).to_dict())
    print(f"KS(매칭거리 vs 무작위거리): stat={stat:.4f} p={p:.3g}")


===== cn7
labeled->unlabeled 최근접거리: {'count': 1211.0, 'mean': 1.309, 'std': 0.137, 'min': 0.841, '25%': 1.224, '50%': 1.307, '75%': 1.401, 'max': 1.897}
무작위 벡터 최근접거리(기준선): {'count': 1211.0, 'mean': 1.404, 'std': 0.154, 'min': 0.777, '25%': 1.313, '50%': 1.415, '75%': 1.51, 'max': 1.834}
KS(매칭거리 vs 무작위거리): stat=0.3039 p=9.53e-50
===== rg3
labeled->unlabeled 최근접거리: {'count': 1182.0, 'mean': 1.281, 'std': 0.164, 'min': 0.825, '25%': 1.166, '50%': 1.29, '75%': 1.403, 'max': 1.704}
무작위 벡터 최근접거리(기준선): {'count': 1182.0, 'mean': 1.387, 'std': 0.161, 'min': 0.825, '25%': 1.279, '50%': 1.398, '75%': 1.502, 'max': 1.854}
KS(매칭거리 vs 무작위거리): stat=0.2623 p=3.9e-36


In [24]:
eps = 0.3
for name, m in [("cn7", match_cn7), ("rg3", match_rg3)]:
    share = float((m["dist"] < eps).mean())
    print(f"{name}: dist<{eps} 인 labeled 행 비율 = {share:.4f} ({int((m['dist']<eps).sum())}/{len(m)})")


cn7: dist<0.3 인 labeled 행 비율 = 0.0000 (0/1211)
rg3: dist<0.3 인 labeled 행 비율 = 0.0000 (0/1182)


**결과**
- labeled→unlabeled 최근접거리 분포(cn7 mean 1.31·min 0.84, rg3 mean 1.28·min 0.83)는 **무작위 벡터의 최근접거리
  분포(cn7 mean 1.40·min 0.78, rg3 mean 1.39·min 0.82)와 거의 같은 범위**에 있고, 오히려 labeled의 최솟값이 무작위
  기준선보다 크거나 비슷하다. KS 검정은 두 분포가 통계적으로 다르다고 나오지만(cn7 stat=0.304, rg3 stat=0.262,
  p≪0.001 — 표본이 커서 작은 차이도 유의하게 나옴) **"거의 완전히 같은 샷"에 해당하는 근접 매칭(ε=0.3)은 cn7·rg3
  모두 0건**이다.
- 즉 **값 기준으로 labeled 행이 unlabeled에 재등장하지 않는다**는 01의 "원본 인덱스가 겹치지 않는다" 관찰이
  근사 매칭으로도 재확인됐다. unlabeled를 사전학습·이상탐지에 쓰더라도 labeled test 행이 값으로 섞여 들어가는
  누수는 (이 근사 기준 안에서는) 없다.

**모델 단계 반영**: unlabeled를 사전학습(오토인코더 등)·이상탐지·드리프트 참고용으로 자유롭게 사용해도 **labeled
평가셋 누수 위험은 낮다**고 판단한다. 다만 이 결론은 "순위 벡터 유클리드 거리 ε=0.3" 기준에서만 유효한 (추정)이므로,
모델 단계에서 실제로 unlabeled 사전학습을 쓴다면 사전학습 전후 labeled CV 성능 차이를 별도로 재확인하는 것을
권장한다.


## ⑤ 변수 성격 전수 점검

**가설**: 01은 `Clamp_Open_Position`이 상수, rg3의 `Injection_Time`(3값)·`Filling_Time`(2값)이 사실상 이산이라고
지적했지만 24변수 × 4파일 전수 점검(고유값 수, 이산/연속, 스파이크 값)은 하지 않았다. rg3 두 변수의 "전환 시점"도
확인이 필요하다.

**실험**: `pairs.var_profile`로 4파일 전체를 점검하고, `pairs.value_segments`로 rg3 `Injection_Time`·`Filling_Time`의
행 순서상 값 전환 구간을 찾아 구간별 불량률을 비교한다.


In [25]:
profiles = {k: pairs.var_profile(dq.split_xy(d)[0]) for k, d in dfs.items()}
n_spike = pd.DataFrame({k: p["is_spike"] for k, p in profiles.items()})
n_discrete = pd.DataFrame({k: p["is_discrete"] for k, p in profiles.items()})
print("=== 파일별 스파이크(top_share>=5%) 변수 개수 ===")
print(n_spike.sum().to_dict())
print("=== 파일별 이산(nunique<=20) 변수 개수 ===")
print(n_discrete.sum().to_dict())
print("=== 상수 변수(nunique==1) ===")
for k, p in profiles.items():
    const_vars = p.index[p["is_constant"]].tolist()
    print(k, const_vars)


=== 파일별 스파이크(top_share>=5%) 변수 개수 ===
{'labeled_cn7': 23, 'labeled_rg3': 23, 'unlabeled_cn7': 22, 'unlabeled_rg3': 23}
=== 파일별 이산(nunique<=20) 변수 개수 ===
{'labeled_cn7': 15, 'labeled_rg3': 15, 'unlabeled_cn7': 0, 'unlabeled_rg3': 0}
=== 상수 변수(nunique==1) ===
labeled_cn7 ['Clamp_Open_Position']
labeled_rg3 ['Clamp_Open_Position']
unlabeled_cn7 []
unlabeled_rg3 []


In [26]:
print("=== rg3 labeled: 이산(고유값<=10) 변수 프로필 ===")
display(profiles["labeled_rg3"][profiles["labeled_rg3"]["nunique"] <= 10].sort_values("nunique"))


=== rg3 labeled: 이산(고유값<=10) 변수 프로필 ===


,nunique,is_constant,is_discrete,top_value,top_share,is_spike
var,,,,,,
Clamp_Open_Position,1,True,True,0.000000,1.000000,False
Filling_Time,2,False,True,0.838552,0.587140,True
Clamp_Close_Time,3,False,True,-0.386790,0.429780,True
Injection_Time,3,False,True,-0.029099,0.986464,True
Average_Screw_RPM,4,False,True,-0.421798,0.583756,True
Max_Screw_RPM,4,False,True,-0.665752,0.428088,True
Max_Injection_Speed,7,False,True,0.617645,0.270728,True
Cushion_Position,7,False,True,0.480326,0.282572,True
Plasticizing_Position,8,False,True,0.588367,0.302876,True


In [27]:
X_rg3_, y_rg3_seg = dq.split_xy(dfs["labeled_rg3"])
seg_it = pairs.value_segments(X_rg3_["Injection_Time"])
seg_ft = pairs.value_segments(X_rg3_["Filling_Time"])
print(f"Injection_Time: 세그먼트 {len(seg_it)}개, 값 {seg_it['value'].unique()}")
print(f"Filling_Time: 세그먼트 {len(seg_ft)}개, 값 {seg_ft['value'].unique()}")
print(seg_it.to_string())


Injection_Time: 세그먼트 15개, 값 [-0.02909945 -8.62773039  8.56963399]
Filling_Time: 세그먼트 108개, 값 [-1.1925314   0.83855234]
    start_pos  end_pos     value  length
0           0       67 -0.029099      68
1          68       71 -8.627730       4
2          72      399 -0.029099     328
3         400      401 -8.627730       2
4         402      953 -0.029099     552
5         954      955  8.569634       2
6         956      967 -0.029099      12
7         968      969  8.569634       2
8         970     1021 -0.029099      52
9        1022     1023  8.569634       2
10       1024     1039 -0.029099      16
11       1040     1041  8.569634       2
12       1042     1115 -0.029099      74
13       1116     1117  8.569634       2
14       1118     1181 -0.029099      64


In [28]:
# Injection_Time: 주값(최다 세그먼트) vs 희귀값 세그먼트의 불량률
main_val = seg_it["value"].mode().iloc[0]
rare = seg_it[seg_it["value"] != main_val]
rows = []
for _, r in seg_it.iterrows():
    sl = y_rg3_seg.iloc[int(r.start_pos):int(r.end_pos) + 1]
    rows.append({"value": r.value, "length": r.length, "n_fail": int(sl.sum()), "fail_rate": float(sl.mean())})
seg_fail = pd.DataFrame(rows)
print("=== Injection_Time 값별(세그먼트 단위) 불량률 ===")
display(seg_fail.groupby("value").agg(n_segments=("length", "size"), total_rows=("length", "sum"),
                                       total_fail=("n_fail", "sum")).assign(
    fail_rate=lambda d: d["total_fail"] / d["total_rows"]))


=== Injection_Time 값별(세그먼트 단위) 불량률 ===


,n_segments,total_rows,total_fail,fail_rate
value,,,,
-8.627730,2,6.0,0,0.000000
-0.029099,8,1166.0,25,0.021441
8.569634,5,10.0,0,0.000000


**결과**
- 파일별 스파이크(한 값이 5% 이상 차지) 변수 수: labeled cn7 23개, labeled rg3 23개, unlabeled cn7 22개, unlabeled
  rg3 23개(24개 중 대부분이 스파이크 후보). z-score 표준화 데이터에서 스파이크가 흔한 것은 **셋포인트 제어(사출
  공정은 기본적으로 설정값을 반복 재현하려 하므로 같은 값이 자주 나옴)**로 설명 가능(추정) — 반드시 숨은 결측을
  의미하지는 않는다.
- 이산 변수(고유값 <=20)로 분류되는 변수는 **labeled cn7·rg3 둘 다 24개 중 15개**(`Injection_Time`, `Filling_Time`,
  `Cycle_Time`, `Clamp_Close_Time`, `Cushion_Position`, `Plasticizing_Position`, `Clamp_Open_Position`,
  `Max_Injection_Speed`, `Max_Screw_RPM`, `Average_Screw_RPM`, `Max_Injection_Pressure`, `Max_Switch_Over_Pressure`,
  `Barrel_Temperature_3`, `Barrel_Temperature_6` + cn7는 `Average_Back_Pressure`/rg3는 `Barrel_Temperature_1`)로
  **동일**하다. 반면 **unlabeled는 두 파일 다 이산 변수가 0개**(모든 변수 고유값 >20) — labeled의 "이산성"이
  unlabeled에서는 재현되지 않는다. 행 수 차이(1,200 대 35,000)만으로 설명하기엔 차이가 너무 크므로, labeled가
  unlabeled보다 **좁은 설정값 범위(또는 다른 수집 조건)**에서 모인 것이라는 기존 가설(②의 idle 블록 부재와 같은
  맥락)을 추가로 뒷받침한다(추정).
- `Clamp_Open_Position`은 4개 파일 전부에서 상수(nunique=1) — 완전한 상수이므로 모델링에서 제거 대상 1순위로 확정.
- rg3 `Injection_Time`은 "단일 전환 시점"이 아니라 **주값(-0.029, top_share 98.6%)과 희귀값(-8.63/+8.57)이 총
  15개 세그먼트(주값 8개 + 희귀값 7개: -8.63 구간 2개·+8.57 구간 5개)로 자주 오가는 반복적 단주기 이벤트**(희귀값
  세그먼트는 대부분 길이 2~4행)다. 당초 가정("전환 전후 불량률 비교")은 이 데이터 구조와 맞지 않아 **세그먼트
  단위 불량률**로 대체 확인했다: 주값(-0.029) 구간 불량률 2.14%(25건/1,166행), 희귀값 구간은 두 값 모두
  **불량 0건**(-8.63: 0/6행, +8.57: 0/10행, 합쳐서 0/16행). `Filling_Time`도 유사하게 108개의 짧은 세그먼트로
  자주 전환된다(단일 전환점 없음).

**모델 단계 반영**: (1) `Clamp_Open_Position`은 4파일 공통 제거. (2) rg3의 `Injection_Time`/`Filling_Time` 희귀값은
불량과 무관(오히려 불량 0건)하므로 "이상치"로 취급해 제거하지 말고, **희귀값 자체를 이진 플래그 변수**(예:
"저해상도 셋포인트 전환 샷 여부")로 남겨 모델이 활용할지 자체적으로 판단하게 한다. (3) 스파이크 변수 다수는
트리 기반 모델(구간 분할에 강함)에는 문제되지 않지만 로지스틱 등 선형 모델에서는 이산화(원-핫 또는 구간화)를
검토한다.


## ⑥ 샷 간 자기상관

**가설**: cn7의 뚜렷한 드리프트(③)를 볼 때 인접 샷 간 값이 서로 비슷할(자기상관이 높을) 가능성이 크다. 높다면
"직전 샷 대비 변화량" 파생 변수가 유효하고, 랜덤 split이 더 위험하다는 뜻이다. 불량이 연속으로 몰리는지(run-length)도
확인이 필요하다.

**실험**: `drift.autocorr_table`로 행 순서(=수집 순서로 간주) 기준 lag-1·lag-5 자기상관을 변수별로 계산한다.
`drift.fail_run_lengths`로 불량 run-length 분포를 본다.


In [29]:
ac_l_cn7 = drift.autocorr_table(X_cn7)
ac_l_rg3 = drift.autocorr_table(X_rg3)
ac_u_cn7 = drift.autocorr_table(dq.split_xy(dfs["unlabeled_cn7"])[0])
ac_u_rg3 = drift.autocorr_table(dq.split_xy(dfs["unlabeled_rg3"])[0])
print("=== lag-1 자기상관 중앙값 ===")
for name, ac in [("labeled_cn7", ac_l_cn7), ("labeled_rg3", ac_l_rg3), ("unlabeled_cn7", ac_u_cn7), ("unlabeled_rg3", ac_u_rg3)]:
    print(f"{name}: lag1 median={ac['lag1'].median():.3f}  lag5 median={ac['lag5'].median():.3f}")
print()
print("=== labeled cn7 lag-1 상위 8 ===")
display(ac_l_cn7.head(8))


=== lag-1 자기상관 중앙값 ===
labeled_cn7: lag1 median=0.859  lag5 median=0.471
labeled_rg3: lag1 median=0.801  lag5 median=0.255
unlabeled_cn7: lag1 median=0.821  lag5 median=0.760
unlabeled_rg3: lag1 median=0.749  lag5 median=0.733

=== labeled cn7 lag-1 상위 8 ===


,lag1,lag5
var,,
Mold_Temperature_4,0.996217,0.981567
Plasticizing_Position,0.995594,0.982985
Mold_Temperature_3,0.995445,0.978605
Clamp_Close_Time,0.975149,0.916456
Max_Back_Pressure,0.965590,0.805570
Plasticizing_Time,0.949180,0.718179
Average_Back_Pressure,0.934802,0.684752
Filling_Time,0.921642,0.595724


In [30]:
fig, ax = plt.subplots(figsize=(10, 5))
d = ac_l_cn7.sort_values("lag1", ascending=False)
x = np.arange(len(d))
ax.bar(x - 0.18, d["lag1"], width=0.36, label="lag-1", color="steelblue")
ax.bar(x + 0.18, d["lag5"], width=0.36, label="lag-5", color="indianred")
ax.set_xticks(x); ax.set_xticklabels(d.index, rotation=80, fontsize=7); ax.legend()
ax.set_title("labeled cn7: 변수별 lag-1/lag-5 자기상관")
save("02_autocorr.png")


saved 02_autocorr.png


In [31]:
print("=== 불량 run-length 분포 ===")
print("cn7:", drift.fail_run_lengths(y_cn7).value_counts().sort_index().to_dict())
print("rg3:", drift.fail_run_lengths(y_rg3).value_counts().sort_index().to_dict())


=== 불량 run-length 분포 ===
cn7: {1: 11, 6: 1}
rg3: {1: 25}


**결과**
- lag-1 자기상관 중앙값이 labeled cn7 **0.859**, unlabeled cn7 **0.821**, labeled rg3 **0.801**, unlabeled rg3
  **0.749**로 **네 파일 전부에서 매우 높다**(변수 절반 이상이 0.7 이상). cn7의 `Barrel_Temperature_6`은 lag-1
  0.95에 달한다. lag-5도 lag-1과 크게 다르지 않아(예: cn7 `Barrel_Temperature_6` lag5=0.93) 자기상관이 5샷 뒤까지도
  잘 유지된다.
- 불량 run-length: cn7은 길이 1(산발) 11건 + 길이 6짜리 연속 run 1건(행 115~120, ③의 드리프트 사건과 동일) —
  "17건의 독립 사건"이 아니라 **사실상 11개의 독립 사건 + 1개의 군집 사건**에 가깝다. rg3는 전부 길이 1(25건 모두
  산발) — 연속 사건이 전혀 없다.

**모델 단계 반영**: (1) 자기상관이 전 파일에서 0.7~0.95 수준으로 매우 높으므로 **"직전 샷 값", "직전 샷 대비 차이"
류의 파생 변수가 유효할 가능성이 높다**(심사 5번 창의성 후보). (2) 이 정도 자기상관에서는 **랜덤 split이 사실상
"미래 정보 누수"와 같다** — 인접 행이 거의 같은 값이므로 랜덤 split은 test 행과 거의 동일한 train 행을 만들어낸다.
GroupKFold(X-키)만으로는 이 문제를 못 막으므로 **시간 블록 split을 사실상 주 평가 방식으로 승격**해야 한다.
(3) cn7 불량은 "11개 독립 + 1개 군집(6연속)"으로 보아 **유효 표본 수를 17이 아니라 약 12건으로 더 보수적으로
잡아야** 한다(합 F1 계산 시 신뢰구간을 과소평가하지 않도록).


## ⑦ cn7·rg3 결합 가능성

**가설**: 01은 cn7·rg3를 "스케일이 달라 그대로 합칠 수 없다"고 지적했지만 z-score 이후 **형태(분포 모양)**가
비슷한지는 비교하지 않았다. 형태가 비슷하면 "금형 플래그 변수 + 단일 모델", 다르면 "별도 모델"이 맞다는 설계
결정(모델 40점)에 직결된다.

**실험**: `drift.shape_compare`로 24개 공통 변수의 KS 통계량·왜도·첨도를 비교한다.


In [32]:
sc = drift.shape_compare(X_cn7, X_rg3)
print("=== cn7 vs rg3 변수별 KS·왜도·첨도 (KS 상위 8) ===")
display(sc.head(8))
print()
print("KS stat 요약:", sc["ks_stat"].describe().round(3).to_dict())
print("KS p<0.001 인 변수 수:", int((sc["ks_p"] < 0.001).sum()), "/", len(sc))


=== cn7 vs rg3 변수별 KS·왜도·첨도 (KS 상위 8) ===


,ks_stat,ks_p,skew_cn7,skew_rg3,kurt_cn7,kurt_rg3
var,,,,,,
Cycle_Time,0.603390,2.447344e-203,4.763385,8.375939,55.993328,96.231617
Average_Screw_RPM,0.575873,5.735702e-184,8.976788,0.250169,128.350260,0.089367
Injection_Time,0.515155,2.569152e-145,-0.807189,2.064315,6.068231,70.754801
Clamp_Close_Time,0.404805,3.445504e-88,0.410294,-0.438498,-1.130038,-0.923900
Plasticizing_Position,0.389154,2.378549e-81,0.767627,-0.037427,-1.231805,-0.587082
Filling_Time,0.384828,1.615448e-79,-0.895344,-0.353979,6.670408,-1.874699
Cushion_Position,0.367709,1.791697e-72,1.014397,0.103606,2.450614,-0.583428
Max_Injection_Speed,0.365411,1.481807e-71,5.265158,-0.020627,46.800757,-0.640533



KS stat 요약: {'count': 24.0, 'mean': 0.268, 'std': 0.159, 'min': 0.0, '25%': 0.159, '50%': 0.229, '75%': 0.372, 'max': 0.603}
KS p<0.001 인 변수 수: 22 / 24


In [33]:
fig, ax = plt.subplots(figsize=(10, 5))
d = sc.sort_values("ks_stat", ascending=False)
ax.barh(d.index[::-1], d["ks_stat"][::-1], color="steelblue")
ax.set_title("cn7 vs rg3: 변수별 KS 통계량(클수록 분포 형태 차이 큼)")
save("02_cn7_vs_rg3_ks.png")


saved 02_cn7_vs_rg3_ks.png


**결과**
- 24개 공통 변수 중 **22개(92%)** 가 KS p<0.001로 분포 형태가 통계적으로 다르다. 예외 2개는 `Clamp_Open_Position`
  (양쪽 다 상수라 KS stat=0)과 `Barrel_Temperature_4`(p=0.082로 유의하지 않음)뿐이다. KS 통계량 중앙값이 0.229로
  꽤 크고, 상위 변수(`Cycle_Time` 0.603, `Average_Screw_RPM` 0.576, `Injection_Time` 0.515)는 분포 모양 자체가
  크게 다르다.
- 왜도·첨도 차이도 극단적이다: 예를 들어 `Average_Screw_RPM`의 첨도가 cn7 128.4 vs rg3 0.09, `Injection_Time`
  첨도가 cn7 6.1 vs rg3 70.8 — 어느 파일이 "더 뾰족한지"조차 변수마다 뒤바뀐다. 단순한 스케일 차이가 아니라
  **금형/설비별로 서로 다른 공정 특성(설정값 반복 패턴, 이산화 수준)을 반영**하는 것으로 보인다(⑤에서 확인한 rg3의
  낮은 해상도·잦은 세트포인트 반복과도 일치).

**모델 단계 반영**: z-score 표준화 이후에도 두 금형의 분포 형태가 근본적으로 다르므로, **"금형 플래그 변수 +
단일 모델"보다 "cn7·rg3 별도 모델"을 기본안으로 확정**한다. 굳이 합쳐서 학습량을 늘리고 싶다면 트리 기반 모델+
금형 플래그 조합으로 한 번 더 검증해볼 수 있으나, 이번 진단 결과만으로는 결합의 이점보다 위험(서로 다른 분포를
한 결정 경계로 학습)이 커 보인다.


## ⑧ 이상치 유형 규칙 코드화

**가설**: 01이 제안한 "공정 이상 / 로그·센서 이상 / 비가동 / 정상" 4분류를 실제 규칙으로 코드화하면, 파일별 유형
비율과 함께 "공정 이상 규칙"만으로 불량을 얼마나 잡아낼 수 있는지(정밀도·재현율) 확인할 수 있다 — 심사 2번(모델)의
베이스라인 후보다.

**실험**: `outliers.classify_outliers`(idle > process(4변수 중 2개 이상 |z|>3) > log(어떤 변수든 |z|>5) > normal
우선순위 규칙)를 4파일에 적용하고, labeled 파일에서 `outliers.rule_precision_recall`로 규칙 기반 불량 예측 성능을
확인한다.


In [34]:
cat_l_cn7 = outliers.classify_outliers(dfs["labeled_cn7"])
cat_l_rg3 = outliers.classify_outliers(dfs["labeled_rg3"])
idle_u_cn7 = dq.idle_block_mask(dfs["unlabeled_cn7"])
idle_u_rg3 = dq.idle_block_mask(dfs["unlabeled_rg3"])
cat_u_cn7 = outliers.classify_outliers(dfs["unlabeled_cn7"], idle_mask=idle_u_cn7)
cat_u_rg3 = outliers.classify_outliers(dfs["unlabeled_rg3"], idle_mask=idle_u_rg3)
print("=== 파일별 이상치 유형 비율 ===")
type_share = pd.DataFrame({
    "labeled_cn7": cat_l_cn7.value_counts(normalize=True),
    "labeled_rg3": cat_l_rg3.value_counts(normalize=True),
    "unlabeled_cn7": cat_u_cn7.value_counts(normalize=True),
    "unlabeled_rg3": cat_u_rg3.value_counts(normalize=True),
}).fillna(0).round(4)
type_share


=== 파일별 이상치 유형 비율 ===


,labeled_cn7,labeled_rg3,unlabeled_cn7,unlabeled_rg3
outlier_type,,,,
idle,0.0000,0.0000,0.5151,0.3660
log,0.0050,0.0220,0.0012,0.0001
normal,0.9868,0.9763,0.4838,0.6339
process,0.0083,0.0017,0.0000,0.0000


In [35]:
fig, ax = plt.subplots(figsize=(8, 4.5))
type_share.T.plot(kind="bar", stacked=True, ax=ax, color=["#c44e52", "#dd8452", "#4c72b0", "#8c8c8c"])
ax.set_title("파일별 이상치 유형 비율"); ax.set_ylabel("비율"); ax.tick_params(axis="x", rotation=20)
ax.legend(title="유형", bbox_to_anchor=(1.02, 1), loc="upper left")
save("02_outlier_type_share.png")


saved 02_outlier_type_share.png


In [36]:
pr_cn7 = outliers.rule_precision_recall(dfs["labeled_cn7"], cat_l_cn7, positive_type="process")
pr_rg3 = outliers.rule_precision_recall(dfs["labeled_rg3"], cat_l_rg3, positive_type="process")
print("=== 규칙 기반(outlier_type=='process') 불량 예측 정밀도/재현율 ===")
print("cn7:", pr_cn7)
print("rg3:", pr_rg3)


=== 규칙 기반(outlier_type=='process') 불량 예측 정밀도/재현율 ===


cn7: {'precision': 0.6, 'recall': 0.35294117647058826, 'f1': 0.4444444444444444, 'n_pred_pos': 10, 'n_true_pos': 17, 'n_correct': 6}
rg3: {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'n_pred_pos': 2, 'n_true_pos': 25, 'n_correct': 0}


**결과**
- 유형 비율: unlabeled는 cn7 51.5%·rg3 36.6%가 `idle`(②의 완전 고정 기준)이고 나머지 대부분은 `normal`, `log`는
  극소수(cn7 41행, rg3 4행)다. labeled는 `idle`이 0(완전 고정 기준으로는 앞서 ②에서 본 rg3 2건은 "6개 전부 일치"인데
  이번 분류는 `dq.idle_block_mask` 기본 정의를 그대로 썼으므로 일치)이고, cn7은 `process` 10행(0.83%)·`log` 6행
  (0.50%), rg3는 `process` 2행(0.17%)·`log` 26행(2.20%)이다.
- 규칙 기반 정밀도/재현율(`process` 유형 → 불량 예측): **cn7 precision=0.60, recall=0.35, F1=0.44**(10건 예측 중
  6건 적중, 실제 불량 17건 중 6건 포착). **rg3는 precision=0.0, recall=0.0**(`process` 예측 2건 모두 실제로는
  양품) — rg3는 이 규칙이 전혀 작동하지 않는다.
- rg3에서 규칙이 실패하는 이유는 ①·③에서 이미 확인했다: rg3 불량은 **피처와 무관한 라벨 충돌**이 대부분(25/25)이라
  어떤 규칙을 세워도 원리적으로 못 잡는다.

**모델 단계 반영**: (1) 이 규칙(F1 0.44, cn7 한정)을 **cn7 전용 베이스라인 0호**로 리포트에 명시하고, 심사 2번의
"베이스라인 포함 2개 이상 모델 비교"에서 로지스틱/트리 모델과 나란히 제시한다. (2) rg3에는 이 규칙을 적용하지
않는다(정밀도·재현율 0으로 무의미) — rg3는 처음부터 "규칙/모델로 설명 안 되는 라벨 노이즈가 지배적"이라는 사실을
그대로 리포트에 반영해 무리한 성능 주장을 피한다. (3) unlabeled의 `idle` 마스크는 전처리 1순위 필터로 그대로
사용한다.


## 종합 — 02 추가 진단이 01 대비 갱신한 것

| 01의 주장 | 02 검증 결과 |
|---|---|
| labeled 모든 X가 정확히 2회, 8자리 반올림 키 | 반올림 없이도 완전 일치 확인(①) |
| labeled에는 비가동 블록 없음 | rg3에 완전 고정 2행(0.17%) 존재 — 부분 정정(②) |
| Mold_Temperature AUC 0.89는 "추정상" 시간 누수성 | 시간블록 AUC 0.61(누수 약 0.29)로 정량 확인(③) |
| 충전 계열 동시극단 10행 중 6행 불량 = 공정 이상 신호 | 6행만 불량구간과 동일 사건, 4행은 파일 끝 별개 이벤트로 불량과 무관 — 절반만 성립(③) |
| labeled/unlabeled 원본 인덱스 안 겹침 | 순위 기반 근사 매칭으로도 근접 매칭 0건 확인 — 값 기준 누수 없음(④) |
| cn7 드리프트 큼(rg3는 상대적으로 적음이라는 뉘앙스) | rg3도 다수 변수에서 강한 드리프트(PSI>5) — 다만 불량과 무관(③) |
| (미언급) 자기상관 | lag-1 중앙값 0.75~0.86으로 전 파일 매우 높음 — 랜덤 split 위험 재확인(⑥) |
| (미언급) cn7·rg3 결합 가능성 | 24변수 전부 KS p<0.001, 왜도/첨도 극단적으로 다름 — 별도 모델 권장(⑦) |

### 모델 단계 반영 사항 요약

| 항목 | 결정 |
|---|---|
| 타깃 정의 | `max`(그룹 내 하나라도 불량이면 양성) 기본. `충돌 제외`는 rg3에서 양성 0개로 사용 불가, cn7도 표본 과소로 참고용에 그침 |
| split 방식 | GroupKFold(X-키) + 시간 블록 split(예: 첫 1/5을 test) **병행 필수**, 랜덤 split 단독 사용 금지(자기상관·드리프트 모두 높음) |
| 결합 여부 | cn7·rg3 **별도 모델** 기본안(형태 차이 KS p<0.001 전 변수) |
| 제거 규칙 | `Clamp_Open_Position`(4파일 공통 상수) 제거. unlabeled idle 마스크(6마커 전부 고정) 유지 |
| 유지 규칙 | labeled 부분 고정 행(rg3 2건 등)은 제거 대신 "비가동 근접도" 플래그로 보존 |
| 파생 변수 후보 | 직전 샷 대비 변화량(lag-1 차분, 자기상관 0.7~0.95로 유효 가능성 높음), 마커 일치 개수/비가동 근접도, rg3 Injection·Filling_Time 희귀 세그먼트 플래그, cn7 시간블록 상대위치 |
| 베이스라인 | 규칙 기반(`outlier_type=='process'`) cn7 precision 0.60/recall 0.35/F1 0.44 — cn7 한정 0호 베이스라인으로 채택, rg3는 부적용 |
